In [114]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score


In [115]:
df = pd.read_csv("./find-polynomial/train.csv")
x1 = df[["x1"]].values.squeeze(axis=1)
x2 = df[["x2"]].values.squeeze(axis=1)
y = df["y"].values

In [116]:
# Feature engineering
X = np.column_stack([
    x1**4,
    x2**4,
    x1**3*x2**1,
    x1**1*x2**3,
    x1**2*x2**2,
    x2**4*x1**1,
    x1**2 * x2**3,
    x1**5,
    x2**5,
    x1**4 * x2**1,
    x1**3 * x2**2,
])
# ===========================
# 1. Add bias column (theta_0)
# ===========================
X_bias = np.hstack([np.ones((X.shape[0], 1)), X])  # shape: (m, n+1)

X_train, X_test, y_train, y_test = train_test_split(X_bias, y, test_size=0.05, random_state=42)

In [117]:
# ===========================
# 2. Compute theta using normal equation
# ===========================
XTX = X_train.T @ X_train

# Check if XTX is invertible
if np.linalg.matrix_rank(XTX) == XTX.shape[0]:
    theta = np.linalg.inv(XTX) @ X_train.T @ y_train
else:
    # Use pseudo-inverse if singular
    theta = np.linalg.pinv(X_train) @ y_train

print("Theta (coefficients including bias):", theta)

Theta (coefficients including bias): [ 7.41562886e-07 -7.85695264e-02 -1.78812939e-03 -1.00412684e-01
  8.07957918e-02  5.31415703e-02  2.53085686e-02 -1.99880432e+00
  1.97685690e+00 -2.32735993e-02  1.51424912e-02 -2.99420952e+00]


In [118]:
# ===========================
# 3. Make predictions
# ===========================
y_pred = X_test @ theta
print("Predicted y:", y_pred)

Predicted y: [-8.67255388e+09 -6.17490714e+09 -1.26915262e+08 -8.37719369e+08
 -1.23428755e+08 -3.40813461e+09 -1.04686584e+06 -3.46989833e+08
 -1.70113962e+10 -1.49062695e+08 -1.24600294e+08 -6.89971518e+05
 -4.11688353e+09 -1.22248554e+08 -5.27056410e+05 -1.80655988e+07
 -4.39866068e+05 -7.42918764e+06 -2.92180631e+09 -7.33889586e+08
 -5.52863960e+09 -7.77076747e+08 -3.45531862e+09 -1.14127114e+10
 -2.66774485e+06 -1.55466571e+08 -3.93609263e+05 -4.22694098e+09
 -1.47964450e+10 -1.59245192e+09 -2.59183895e+06 -1.08837460e+08
 -1.80968436e+09 -1.01941131e+07 -1.50112681e+08 -1.57793187e+08
 -8.28734031e+09 -2.06144633e+07 -2.39280820e+10 -1.40632928e+10]


In [119]:
print("=== Linear Features ===")
print("MSE:", mean_squared_error(y_test, y_pred))

=== Linear Features ===
MSE: 3.589284760686309


In [120]:
df_test = pd.read_csv("./find-polynomial/test.csv")
df_test.head()

,x1,x2
0,70.120120,114.144144
1,70.270270,114.324324
2,70.420420,114.504505
3,70.570571,114.684685
4,70.720721,114.864865


In [121]:
x1f = df_test[["x1"]].squeeze(1)
x2f = df_test[["x2"]].squeeze(1)
# Feature engineering
X_test_f = np.column_stack([
    x1f**4,
    x2f**4,
    x1f**3*x2f**1,
    x1f**1*x2f**3,
    x1f**2*x2f**2,
    x2f**4*x1f**1,
    x1f**2 * x2f**3,
    x1f**5,
    x2f**5,
    x1f**4 * x2f**1,
    x1f**3 * x2f**2,
])

X_test_f = np.hstack([np.ones((X_test_f.shape[0], 1)), X_test_f])

In [122]:
y_test_f = X_test_f @ theta
y_test_f.shape

(200,)

In [123]:
submission = pd.DataFrame(np.vstack([np.arange(200),y_test_f]).T,columns=["id","y"])
submission[["id"]] = submission[["id"]].astype(int)
submission.to_csv('./submission_4.csv', index=False)